# Preclinical translation — Phase D

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

Phase D adds the discovery-to-clinic bridge: the **Simeoni 2004** preclinical xenograft model (exponential→linear growth + a signal-distribution transit chain giving *delayed* cell death) and an **in-vitro → in-vivo potency translation**. The Simeoni model is the project's first multi-state ODE system; the kernel framework integrates it numerically and exports all four compartments to SBML/NONMEM.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.export.registry import get_kernel

ds = onkos.load()
spec = get_kernel(ds['preclinical_translation.simeoni_2004.xenograft'])
print('Simeoni states:', spec.states, '| observable:', spec.observable, '| analytic:', spec.analytic)

In [ ]:
# Unperturbed growth is exponential (rate ~lambda0) then linear (rate ~lambda1).
t = np.linspace(0, 60, 241)  # days
g = onkos.simulate(ds, 'growth_laws.simeoni_exp_linear', context={'tumor_type': 'ovarian_xenograft'}, drug_effect=0.0, t=t)
w = g.tumor_size
early = (np.log(w[10]) - np.log(w[2])) / (t[10] - t[2])
late = (w[-1] - w[-10]) / (t[-1] - t[-10])
print(f'early log-slope ~lambda0: {early:.3f} (expect ~0.20)')
print(f'late linear-slope ~lambda1: {late:.3f} (expect ~0.70)')
assert abs(early - 0.2) < 0.03 and abs(late - 0.7) < 0.1
plt.semilogy(t, w); plt.xlabel('days'); plt.ylabel('w (g, log)'); plt.title('Simeoni exp->linear');

In [ ]:
# Drug damages proliferating cells; damaged cells traverse x2->x3->x4 before dying.
rid = 'preclinical_translation.simeoni_2004.xenograft'
ctx = {'tumor_type': 'ovarian_xenograft'}
td = np.linspace(0, 40, 161)
for dose in [0.0, 50.0, 120.0]:
    tr = onkos.simulate(ds, rid, context=ctx, drug_effect=dose, t=td)
    plt.semilogy(td, tr.tumor_size, label=f'E={dose}')
plt.legend(); plt.xlabel('days'); plt.ylabel('w (g, log)'); plt.title('Dose-dependent TGI');

In [ ]:
# Preclinical models carry no survival curve and are excluded from the clinical view.
tr = onkos.simulate(ds, rid, context=ctx, drug_effect=100.0)
print('os_curve:', tr.os_curve)
# Applying xenograft params to a human tumor floors to D (the translation gap).
human = onkos.simulate(ds, rid, context={'tumor_type': 'NSCLC', 'line': 'first'}, drug_effect=100.0)
print('xenograft-on-human tier:', human.tier)
assert human.tier == 'D'

# In-vitro -> in-vivo potency translation (power law).
from onkos.export.reference import effect
from onkos.export.registry import kernel_values
iv = ds['preclinical_translation.ivive_potency']
for ic50 in [5, 20, 100]:
    print(f'IC50={ic50} ng/mL -> in-vivo potency {float(effect(get_kernel(iv), ic50, kernel_values(iv))):.2f}')